# Home Credit Default Risk — Error Analysis

This notebook contains false positive rate audits at specified recall targets, analyzes prediction overlaps/misclassifications between tree and neural network models, and evaluates model subgroup fairness.

## 1. Setup & Imports

In [1]:
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_auc_score, confusion_matrix, precision_recall_curve

# Append parent directory to path to enable src imports
sys.path.append("..")
# Change working directory to project root so relative paths work
os.chdir("..")

from src.models.train import load_data
from src.features.application import build_application_features

%matplotlib inline
sns.set_theme(style="whitegrid")

## 2. Load Evaluation Data & Best Model Probabilities

In [2]:
_, _, _, _, X_test, y_test = load_data()

import mlflow
from mlflow.tracking import MlflowClient

mlflow.set_tracking_uri("mlruns")
client = MlflowClient()

experiments = client.search_experiments()
exp_ids = [e.experiment_id for e in experiments]

predictions = {}
model_types = ["xgboost", "entity_embedding"]

for m_type in model_types:
    runs = client.search_runs(
        experiment_ids=exp_ids,
        filter_string=f"tags.model_type = '{m_type}'",
        order_by=["metrics.test_roc_auc DESC"],
        max_results=1
    )
    if runs:
        run_id = runs[0].info.run_id
        ext = ".pt" if m_type == "entity_embedding" else ".pkl"
        model_path = os.path.join("mlflow_artifacts", f"model_{m_type}_{run_id}{ext}")
        if os.path.exists(model_path):
            if m_type == "xgboost":
                from src.models.xgboost_model import XGBoostModel
                model = XGBoostModel.load(model_path)
            else:
                from src.models.entity_embedding import EntityEmbeddingModel
                model = EntityEmbeddingModel.load(model_path)
            predictions[m_type] = model.predict_proba(X_test)

Loading existing split indices from data/splits...
Dataset splits loaded successfully:
  Train: X=(400, 77), y=(400,), positive_rate=7.00%
  Val:   X=(50, 77), y=(50,), positive_rate=12.00%
  Test:  X=(50, 77), y=(50,), positive_rate=10.00%


## 3. False Positive Rate (FPR) at Recall Thresholds
Evaluate FPR at 10%, 20%, 30% recall levels to see how many safe applicants are rejected to catch a fraction of defaults.

In [3]:
recalls_target = [0.10, 0.20, 0.30]

for name, probs in predictions.items():
    print(f"\n=== {name} FPR at Recall levels ===")
    prec, rec, thresholds = precision_recall_curve(y_test, probs)
    
    for target in recalls_target:
        # Find the threshold closest to target recall
        idx = np.argmin(np.abs(rec - target))
        thresh = thresholds[min(idx, len(thresholds)-1)]
        
        # Apply threshold
        preds_bin = (probs >= thresh).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_test, preds_bin).ravel()
        fpr = fp / (fp + tn)
        actual_rec = tp / (tp + fn)
        print(f"  Target Recall: {target*100:.0f}% -> Actual Recall: {actual_rec*100:.2f}% | FPR: {fpr*100:.2f}%")


=== xgboost FPR at Recall levels ===
  Target Recall: 10% -> Actual Recall: 20.00% | FPR: 24.44%
  Target Recall: 20% -> Actual Recall: 20.00% | FPR: 24.44%
  Target Recall: 30% -> Actual Recall: 20.00% | FPR: 24.44%

=== entity_embedding FPR at Recall levels ===
  Target Recall: 10% -> Actual Recall: 20.00% | FPR: 2.22%
  Target Recall: 20% -> Actual Recall: 20.00% | FPR: 2.22%
  Target Recall: 30% -> Actual Recall: 20.00% | FPR: 2.22%


## 4. Misclassification Overlaps
Find how many defaults were missed (False Negatives) or good loans rejected (False Positives) by BOTH models vs uniquely by one.

In [4]:
if "xgboost" in predictions and "entity_embedding" in predictions:
    y_true = y_test.values
    xgb_preds = (predictions["xgboost"] >= 0.1).astype(int) # using 10% threshold as decision boundary
    een_preds = (predictions["entity_embedding"] >= 0.1).astype(int)
    
    # False Negatives
    xgb_fn = set(np.where((y_true == 1) & (xgb_preds == 0))[0])
    een_fn = set(np.where((y_true == 1) & (een_preds == 0))[0])
    
    print("\n=== Default Miss (FN) Overlap ===")
    print(f"Total Defaults in Test Set: {(y_true == 1).sum()}")
    print(f"XGBoost Missed:             {len(xgb_fn)}")
    print(f"EEN Missed:                 {len(een_fn)}")
    print(f"Both Missed:                {len(xgb_fn.intersection(een_fn))}")
    print(f"Only XGBoost Missed:        {len(xgb_fn - een_fn)}")
    print(f"Only EEN Missed:            {len(een_fn - xgb_fn)}")


=== Default Miss (FN) Overlap ===
Total Defaults in Test Set: 5
XGBoost Missed:             5
EEN Missed:                 0
Both Missed:                0
Only XGBoost Missed:        5
Only EEN Missed:            0


## 5. Subgroup Analysis (Fairness Audit)
Evaluate performance across different groups (e.g. gender proxy columns or quartiles of income).

In [5]:
# Load original columns to map back demo details (CODE_GENDER is binary encoded 0/1 in processed data)
# Let's check gender feature values in X_test
if "CODE_GENDER" in X_test.columns:
    for name, probs in predictions.items():
        print(f"\n=== {name} Subgroup AUCs ===")
        for gender_val in [0, 1]:
            mask = X_test["CODE_GENDER"] == gender_val
            if mask.sum() > 0 and len(np.unique(y_test[mask])) > 1:
                auc = roc_auc_score(y_test[mask], probs[mask])
                print(f"  CODE_GENDER {gender_val} AUC: {auc:.4f} (n={mask.sum()})")


=== xgboost Subgroup AUCs ===
  CODE_GENDER 0 AUC: 0.6774 (n=33)
  CODE_GENDER 1 AUC: 0.5238 (n=17)

=== entity_embedding Subgroup AUCs ===
  CODE_GENDER 0 AUC: 0.7419 (n=33)
  CODE_GENDER 1 AUC: 0.4286 (n=17)
